# Config

In [1]:
!pip install optuna==4.5.0
!pip install plotly==6.3.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
import json

# 1) SPECTER OPTUNA

### Preprocess

In [4]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


### Optuna

In [ ]:
from transformers import logging
import warnings
import optuna
from pipelines.fine_tune_models import optuna_objective_cv, convert_numpy_to_native

warnings.filterwarnings(
    "ignore",
    message="TypedStorage is deprecated"
)
# Desactiva solo los warnings
logging.set_verbosity_error()

#Definir modelo para realizar fine tuning
model_name = "allenai/specter"

# ----------- Lanzar la optimización -----------
results_dir = "/tmp/results/finetune/SPECTER"
os.makedirs(results_dir, exist_ok=True)
#Crear estudio de optuna
study = optuna.create_study(
    direction="maximize",
    #study_name="specter",
    #storage= f"sqlite:///{results_dir}/optuna_3.db",
    #load_if_exists=True  # evita sobreescribir si ya existe
)
X_train=np.array(X_train)

#Crear función objetivo
opt_model = optuna_objective_cv(X_train, y_train, df_decode=df_decode, n_classes=2, model_name = model_name,
                                sample_weights_loss=True, Test_mode=True)
#Optimización
n_trials=1
study.optimize(opt_model.objective, n_trials=n_trials)

# ----------- Mostrar mejores resultados -----------
print("Mejor f1-score:", study.best_value)
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

#Read results from the best model
results_best_model = opt_model.get_results()

# Aplica la conversión
metrics_native = convert_numpy_to_native(results_best_model['metrics'])

# Imprime con formato limpio
print({'metrics': metrics_native})

In [ ]:
import optuna
from optuna.visualization import plot_param_importances, plot_contour
import matplotlib.pyplot as plt

# ---------- 1. Importancia de Hiperparámetros ----------
fig1 = plot_param_importances(study)
fig1.show()

# ---------- 2. Gráfico de Contorno 2D ----------
# Encuentra los 2 hiperparámetros más importantes
importances = optuna.importance.get_param_importances(study)
top_params = list(importances.keys())[:2]

plot_contour(study, params=["lr", "n_unfreeze"])
    

### Retrain and eval model with the best params

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pipelines.fine_tune_models import Pytorch_Pipeline, mlflow_ckeckpoint
from utils.dataset import CvCustom, TextDataset
from torch.utils.data import DataLoader

#Definir variables
model_name = "allenai/specter"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

#Sin optuna
params={
    "lr": 3.452088271232921e-05,
    "batch_size":5,
    "n_unfreeze":12 #12 max
    }

extra_parms={
    "n_trials":n_trials
}
#Reentrenar con mejores hyperparámetros definidos por optuna
#params=study.best_params

#Train
results = []
for nfold, (train_idx, test_idx) in enumerate(cv_function.split(X_train)):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=None, max_epochs=200)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])
    #MLflow
    pipeline.update_to_best_model() #The principal model will be the best model on validation set
    exp_info={
        "exp_name": "SPECTER_finetuning",
        "run_name":f"fold{nfold}"
    }
    #mlflow_ckeckpoint(exp_info, pipeline, extra_parms, test_loader, y_test, df_test, mode="server")


mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)

# 2) RoBERTa

### Preprocess

In [4]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

### Optuna

In [ ]:
from transformers import logging
import warnings
import optuna
from pipelines.fine_tune_models import optuna_objective_cv, convert_numpy_to_native

warnings.filterwarnings(
    "ignore",
    message="TypedStorage is deprecated"
)
# Desactiva solo los warnings
logging.set_verbosity_error()

#Definir modelo para realizar fine tuning
model_name = "roberta-large"

# ----------- Lanzar la optimización -----------
results_dir = "/tmp/results/finetune/RoBERTa"
os.makedirs(results_dir, exist_ok=True)
#Crear estudio de optuna
study = optuna.create_study(
    direction="maximize",
    study_name="Roberta_ft",
    storage= f"sqlite:///{results_dir}/optuna_0.db",
    load_if_exists=True  # evita sobreescribir si ya existe
)
X_train=np.array(X_train)

#Crear función objetivo
opt_model = optuna_objective_cv(X_train, y_train, df_decode=df_decode, n_classes=2, model_name = model_name,
                                sample_weights_loss=True)
#Optimización
n_trials=20
study.optimize(opt_model.objective, n_trials=n_trials)

# ----------- Mostrar mejores resultados -----------
print("Mejor f1-score:", study.best_value)
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

#Read results from the best model
results_best_model = opt_model.get_results()

# Aplica la conversión
metrics_native = convert_numpy_to_native(results_best_model['metrics'])

# Imprime con formato limpio
print({'metrics': metrics_native})

[I 2025-09-02 21:29:49,707] Using an existing study with name 'Roberta_ft' instead of creating a new one.


(771,)
{'accuracy': 0.5758754863813229, 'precision': 0.28793774319066145, 'recall': 0.5, 'f1_score': 0.4208867752317817, 'cm': array([[  0, 109],
       [  0, 148]])}
{'accuracy': 0.42412451361867703, 'precision': 0.21206225680933852, 'recall': 0.5, 'f1_score': 0.2526206119368076, 'cm': array([[109,   0],
       [148,   0]])}


### Retrain

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pipelines.fine_tune_models import Pytorch_Pipeline, mlflow_ckeckpoint
from utils.dataset import CvCustom, TextDataset
from torch.utils.data import DataLoader

#Definir variables
model_name = "roberta-base"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

#Sin optuna
params={
    "lr": 3.452088271232921e-05,
    "batch_size":5,
    "n_unfreeze":12 #12 max
    }


extra_parms={
    "n_trials":n_trials
}
#Reentrenar con mejores hyperparámetros definidos por optuna
#params=study.best_params

#Train
results = []
for nfold, (train_idx, test_idx) in enumerate(cv_function.split(X_train)):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=None, max_epochs=1)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])
    #MLflow
    pipeline.update_to_best_model() #The principal model will be the best model on validation set
    exp_info={
        "exp_name": "SPECTER_finetuning",
        "run_name":f"fold{nfold}"
    }
    #mlflow_ckeckpoint(exp_info, pipeline, extra_parms, test_loader, y_test, df_test, mode="server")


mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)

(771,)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


f1: 0.5543638986074327
{'accuracy': 0.6476683937823834, 'precision': 0.6901515151515152, 'recall': 0.5965172489562733, 'f1_score': 0.5677206851119894, 'cm': array([[ 21,  61],
       [  7, 104]])}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


f1: 0.4208867752317817
{'accuracy': 0.5751295336787565, 'precision': 0.28756476683937826, 'recall': 0.5, 'f1_score': 0.3651315789473684, 'cm': array([[  0,  82],
       [  0, 111]])}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


f1: 0.47947674775325705
{'accuracy': 0.6062176165803109, 'precision': 0.7967914438502675, 'recall': 0.5365853658536586, 'f1_score': 0.44066503965832826, 'cm': array([[  6,  76],
       [  0, 111]])}
mean: 0.45783910123922866
std: 0.0835934518070781
